# CHB Pipeline — LLM Response Generation (Resampled)
## Llama 3.1 8B Instruct (4-bit) on Tesla T4

**Upload:** `facilitator_moves_resampled.csv`
**Download:** `llm_responses_resampled.csv`

### Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Accept license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
3. Get HF token at https://huggingface.co/settings/tokens
4. Run all cells in order (~30-50 min)


In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes torch pandas huggingface_hub tqdm


In [ ]:
# Cell 2: Login to HuggingFace
from huggingface_hub import login
login()


In [ ]:
# Cell 3: Upload data
from google.colab import files
import pandas as pd

print("Upload: facilitator_moves_resampled.csv")
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_csv(fname)
print(f"\nLoaded {len(df)} moves")
for phase, cnt in df["session_phase"].value_counts().items():
    print(f"  {phase}: {cnt}")


In [ ]:
# Cell 4: Load Llama 3.1 8B Instruct (4-bit quantized)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print(f"Model loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
# Cell 5: Define generation functions
import re, time

SYSTEM_PROMPT = (
    "You are an experienced design thinking facilitator guiding a team "
    "through a product design project. Your role is to ask questions, "
    "reframe problems, and help the team explore new directions -- not "
    "to give direct answers or solutions. Respond with a single "
    "facilitation move of 1-3 sentences only. Do not add meta-commentary "
    "or explanations."
)

REFUSAL_PHRASES = ["as an ai", "i cannot", "i'm unable", "i am unable",
                   "i don't have", "as a language model", "i'm just"]

def truncate_context(ctx, max_words=180):
    words = ctx.split()
    if len(words) <= max_words:
        return ctx
    return "..." + " ".join(words[-max_words:])

def generate_response(context_text, phase, temperature):
    ctx = truncate_context(context_text, max_words=180)
    user_msg = (
        f"Here is an excerpt from a design team meeting:\n"
        f"[CONTEXT]\n{ctx}\n[END CONTEXT]\n\n"
        f"Current design phase: {phase}\n"
        f"Provide your next facilitation move:"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=150,
            temperature=max(temperature, 0.01),
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    response = response.split("\n\n")[0].strip()
    if response.startswith('"') and response.endswith('"'):
        response = response[1:-1].strip()
    return response

def is_valid_response(response, context_text):
    if not response or len(response.split()) < 5:
        return False
    resp_lower = response.lower()
    for phrase in REFUSAL_PHRASES:
        if phrase in resp_lower:
            return False
    ctx_words = set(context_text.lower().split())
    resp_words = set(resp_lower.split())
    if len(resp_words) > 0 and len(ctx_words) > 10:
        overlap = len(resp_words & ctx_words) / len(resp_words)
        if overlap > 0.9:
            return False
    return True

# Quick test
test = generate_response(
    "[facilitator]: What features should we prioritize? "
    "| [designer]: Voice control could be interesting.",
    "Ideate", 0.7
)
print(f"Test: {test}")
print("Model ready!")


In [ ]:
# Cell 6: Generate all responses (~30-50 min)
from tqdm import tqdm
import numpy as np

TEMPERATURES = [0.3, 0.7, 1.0]
MAX_RETRIES = 3

results = []
total = len(df)
start_time = time.time()

print(f"Generating {total} x {len(TEMPERATURES)} = {total*len(TEMPERATURES)} responses...")

for i, (_, row) in enumerate(tqdm(df.iterrows(), total=total)):
    context_id = row["context_id"]
    context_text = str(row["context_text"])
    human_response = str(row["human_response"])
    phase = str(row["session_phase"])

    result = {
        "context_id": context_id,
        "human_response": human_response,
        "session_phase": phase,
        "context_text": context_text,
    }

    for temp in TEMPERATURES:
        key = f"llm_t{str(temp).replace('.', '')}"
        resp = ""
        is_ok = False
        for attempt in range(MAX_RETRIES):
            try:
                resp = generate_response(context_text, phase, temp)
                is_ok = is_valid_response(resp, context_text)
                if is_ok:
                    break
            except Exception as e:
                resp = f"ERROR: {e}"
                is_ok = False

        result[key] = resp
        result[f"{key}_valid"] = is_ok

    results.append(result)

    # Checkpoint every 50
    if (i + 1) % 50 == 0:
        pd.DataFrame(results).to_csv("llm_responses_checkpoint.csv", index=False)
        elapsed = time.time() - start_time
        remaining = (total - i - 1) / ((i + 1) / elapsed)
        print(f"  Checkpoint: {i+1}/{total}, ~{remaining/60:.0f}min left")


In [ ]:
# Cell 7: Save and download
df_results = pd.DataFrame(results)
df_results.to_csv("llm_responses_resampled.csv", index=False)

elapsed_total = time.time() - start_time

print("=" * 60)
print("  GENERATION COMPLETE")
print("=" * 60)
print(f"  Rows:       {len(df_results)}")
print(f"  Time:       {elapsed_total/60:.1f} min")

for tk in ['llm_t03', 'llm_t07', 'llm_t10']:
    nv = df_results[f"{tk}_valid"].sum()
    print(f"  {tk}: {nv}/{len(df_results)} valid ({nv/len(df_results)*100:.1f}%)")

# Samples
print("\nSample responses (T=0.7):")
for idx in range(min(3, len(df_results))):
    r = df_results.iloc[idx]
    print(f"  [{idx+1}] {r['session_phase']}")
    print(f"    Human: {str(r['human_response'])[:100]}")
    print(f"    LLM:   {str(r['llm_t07'])[:100]}")

# Word counts
for tk in ['llm_t03', 'llm_t07', 'llm_t10']:
    wc = df_results[tk].apply(lambda x: len(str(x).split()))
    print(f"  {tk} words: mean={wc.mean():.1f}, min={wc.min()}, max={wc.max()}")

print("\nDownloading llm_responses_resampled.csv...")
files.download("llm_responses_resampled.csv")
print("Place at: design_facilitation_study/data/outputs/llm_responses_resampled.csv")
